In [1]:
# 引入数据分析所需要的库
import numpy as np
import pandas as pd

# 读取数据

**导入数据分析所需要的库，并通过Pandas的`read_csv`函数将原始数据文件"delivery_data.csv"里的数据内容解析为DataFrame，并赋值给变量`original_delivery_data`。**

In [2]:
# 读取数据
original_delivery_data = pd.read_csv("D:/delivery-data-analysis-main/delivery_data.csv")

In [3]:
# 调用head()看前5条数据长什么样
original_delivery_data.head()

,Order_ID,Date,Delivery_Status,Delivery_Time,Delivery_Cost,Customer_Rating,Rider_ID
0,ORDER1,2023-09-27,Failed,119,64.46,4,RIDER1
1,ORDER2,2023-08-21,Failed,73,94.63,4,RIDER2
2,ORDER3,2023-11-22,Failed,49,324.06,3,RIDER3
3,ORDER4,2023-01-17,In Transit,83,463.06,1,RIDER4
4,ORDER5,2023-11-28,Delivered,21,469.34,5,RIDER5


# 评估数据

**在这一部分我将对在上一部分建立的`original_delivery_data`这个DataFrame所包含的数据进行评估。**
**评估主要从两个方面进行：结构和内容，即整齐度和干净度。数据的结构性问题指不符合“每列是一个变量，每行是一个观察值，每个单元格是一个值”这三个标准，数据的内容性问题包括存在丢失数据、重复数据、无效数据等。**

## 评估数据整齐度

In [4]:
# 调用sample()随机抽取10条数据查看
original_delivery_data.sample(10)

,Order_ID,Date,Delivery_Status,Delivery_Time,Delivery_Cost,Customer_Rating,Rider_ID
9576,ORDER9577,2023-01-07,Delivered,59,414.32,4,RIDER9577
5872,ORDER5873,2023-11-21,Delivered,41,231.29,5,RIDER5873
6293,ORDER6294,2023-06-20,In Transit,45,381.47,2,RIDER6294
3295,ORDER3296,2023-12-23,Delivered,32,91.07,2,RIDER3296
7410,ORDER7411,2023-10-22,In Transit,32,457.41,1,RIDER7411
3176,ORDER3177,2023-10-02,Failed,67,173.41,2,RIDER3177
1759,ORDER1760,2023-03-26,Delivered,69,315.40,3,RIDER1760
3216,ORDER3217,2023-01-14,Delivered,50,124.65,1,RIDER3217
4184,ORDER4185,2023-12-13,Failed,13,167.60,4,RIDER4185
2830,ORDER2831,2023-07-26,Failed,114,476.87,2,RIDER2831


**从抽样的10行数据来看，数据符合“每列是一个变量，每行是一个观察值，每个单元格是一个值”。**

## 评估数据干净度

In [5]:
# 调用info()对数据内容进行大致了解
original_delivery_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order_ID         10000 non-null  str    
 1   Date             10000 non-null  str    
 2   Delivery_Status  10000 non-null  str    
 3   Delivery_Time    10000 non-null  int64  
 4   Delivery_Cost    10000 non-null  float64
 5   Customer_Rating  10000 non-null  int64  
 6   Rider_ID         10000 non-null  str    
dtypes: float64(1), int64(2), str(4)
memory usage: 547.0 KB


**从输出结果来看，数据共有10000条观察值，无缺失值。**

1. Date(当前类型: str)
为什么不合适：现在是字符串（str），这意味着计算机把它当成了一堆文字，而不是日期。你无法直接对它进行“按时间排序”、“计算两个日期相差几天”或“提取月份”等操作。
修改建议：转换为日期时间类型 (datetime)。

2. Delivery_Status(当前类型: str)
为什么不合适：这一列只有固定的几个状态（如 "Delivered", "In Transit", "Failed"）。虽然 str也能用，但它占用的内存比专门的“分类类型”要多，而且在统计各个状态的数量时会稍微慢一点。
修改建议：转换为分类类型 (category)。

3. Customer_Rating(当前类型: int64)
为什么不合适：评分通常是像 1, 2, 3, 4, 5 这样有限的整数。虽然用 int64（64位整数）能存下，但对于这种范围很小的数据来说，有点“大材小用”，浪费内存。
修改建议：转换为更小的整数类型，比如 int8（8位整数）。

### 评估缺失数据

**无缺失数据。**

### 评估重复数据

1. 单独列变量（主键/唯一性评估）

我们需要判断：这一列里的数据，是不是天然就应该不重复？

✅ Order_ID(订单编号) —— 绝对不可以重复（核心主键）

理由：这是每笔订单的“身份证号”。在真实业务中，一个 Order_ID只能对应一笔订单。如果出现了两行一模一样的 Order_ID，那肯定是数据录入错误或系统故障。

检查目标：这列必须是 100% 唯一​ 的。

❌ Date(日期) —— 肯定可以重复

理由：每天都有成千上万的订单，所以日期会反复出现（比如有1000个订单都是 2023-01-07）。

❌ Delivery_Status(配送状态) —— 肯定可以重复

理由：成千上万的订单都处于 "Delivered" 状态，这很正常。

❌ Delivery_Time(配送时长) —— 肯定可以重复

理由：可能有很多订单都恰好用了 30 分钟送达。

❌ Delivery_Cost(配送费) —— 肯定可以重复

理由：同一家店的订单，配送费可能完全一样。

❌ Customer_Rating(客户评分) —— 肯定可以重复

理由：很多顾客可能都给了 5 分好评。

⚠️ Rider_ID(骑手编号) —— 可以重复（但有限制）

理由：一个骑手一天会送很多单，所以他的 ID 会出现在多行里。但是，如果某个骑手的单子特别多，可能需要关注是否数据录入有误（比如把同一个订单复制了10次都算在这个骑手头上）。

2. 列变量的组合情况（联合唯一性评估）

我们需要判断：哪几列组合在一起，才代表一个唯一的业务事件？

✅ Order_ID+ Rider_ID(订单编号 + 骑手编号) —— 不可以重复

理由：这不仅确认了是哪笔订单，还确认了是谁送的。通常情况下，一笔订单只能由一个骑手配送。如果这里出现了重复，意味着“同一个订单被同一个骑手记录了两次”，这是严重的重复数据。

⚠️ Order_ID+ Date(订单编号 + 日期) —— 不可以重复

理由：通常一笔订单只会在某一天发生。如果这两个组合起来有重复行，说明同一天同一个订单被记录了两次。

❌ Rider_ID+ Date(骑手编号 + 日期) —— 可以重复

理由：一个骑手在同一天会送很多单，所以组合起来会有很多行。

❌ Delivery_Status+ Delivery_Cost(状态 + 费用) —— 可以重复

理由：很多已送达的订单，费用可能都是 5 块钱。

**1.整行数据完全重复（最明显的垃圾数据）：**

In [7]:
# 检查是否有完全一模一样的行
duplicate_rows = original_delivery_data[original_delivery_data.duplicated()]
# 输出结果
duplicate_rows

,Order_ID,Date,Delivery_Status,Delivery_Time,Delivery_Cost,Customer_Rating,Rider_ID


**从输出结果来看，不存在整行数据完全重复的情况。**

**2.核心主键 Order_ID重复（最严重的业务错误）：**

In [8]:
# 检查 Order_ID 是否有重复，keep=False 会把所有重复的都显示出来
duplicate_orders = original_delivery_data[original_delivery_data['Order_ID'].duplicated(keep=False)] 
# 输出结果
duplicate_orders

,Order_ID,Date,Delivery_Status,Delivery_Time,Delivery_Cost,Customer_Rating,Rider_ID


**从输出结果来看，Order_ID变量不存在重复数据。**

**3.Rider_ID可以重复，但有限制**

In [9]:
# 检查 Rider_ID 是否有重复，keep=False 会把所有重复的都显示出来
duplicate_riders = original_delivery_data[original_delivery_data['Rider_ID'].duplicated(keep=False)]
# 输出结果
duplicate_riders

,Order_ID,Date,Delivery_Status,Delivery_Time,Delivery_Cost,Customer_Rating,Rider_ID


**从输出结果来看，Rider_ID变量不存在重复数据。**

**4.Order_ID+ Rider_ID(订单编号 + 骑手编号) —— 不可以重复**

In [10]:
# 方法1：标记并查看所有重复的行
# 使用 subset 指定要检查的列
duplicate_mask = original_delivery_data.duplicated(subset=['Order_ID', 'Rider_ID'], keep=False)

# 获取所有重复行
duplicate_rows = original_delivery_data[duplicate_mask]

# 查看重复了多少对
print(f"Order_ID + Rider_ID 重复的组合数量: {len(duplicate_rows)}")

# 查看具体哪些重复了
if len(duplicate_rows) > 0:
    print("重复的行：")
    print(duplicate_rows.sort_values(['Order_ID', 'Rider_ID']))

Order_ID + Rider_ID 重复的组合数量: 0


**从输出结果来看，Order_ID+ Rider_ID(订单编号 + 骑手编号)不存在重复数据。**

**5.Order_ID+ Date(订单编号 + 日期) —— 不可以重复**

In [11]:
# 方法1：标记并查看所有重复的行
# 使用 subset 指定要检查的列
duplicate_mask = original_delivery_data.duplicated(subset=['Order_ID', 'Date'], keep=False)

# 获取所有重复行
duplicate_rows = original_delivery_data[duplicate_mask]

# 查看重复了多少对
print(f"Order_ID + Date 重复的组合数量: {len(duplicate_rows)}")

# 查看具体哪些重复了
if len(duplicate_rows) > 0:
    print("重复的行：")
    print(duplicate_rows.sort_values(['Order_ID', 'Date']))

Order_ID + Date 重复的组合数量: 0


**从输出结果来看，Order_ID+ Date(订单编号 + 日期)不存在重复数据。**

### 评估不一致数据

1.逻辑不一致（业务逻辑矛盾）

这是最常见的问题，即数据在业务逻辑上说不通。

疑点：配送状态与时长/评分的矛盾

逻辑：样本中有 Failed（配送失败）的状态。通常，失败的订单不应该有“客户评分”（或者评分极低，如1星），且配送时长可能异常（比如卡在某一状态很久，或者时长为0/空）。

检查方法：Python

**找出所有配送失败的订单**

In [12]:
# 筛选配送失败的订单
failed_orders = original_delivery_data[original_delivery_data['Delivery_Status'] == 'Failed']
# 输出结果
failed_orders

,Order_ID,Date,Delivery_Status,Delivery_Time,Delivery_Cost,Customer_Rating,Rider_ID
0,ORDER1,2023-09-27,Failed,119,64.46,4,RIDER1
1,ORDER2,2023-08-21,Failed,73,94.63,4,RIDER2
2,ORDER3,2023-11-22,Failed,49,324.06,3,RIDER3
8,ORDER9,2023-09-19,Failed,112,277.74,2,RIDER9
10,ORDER11,2023-05-24,Failed,83,175.16,1,RIDER11
...,...,...,...,...,...,...,...
9991,ORDER9992,2023-09-28,Failed,83,63.14,4,RIDER9992
9993,ORDER9994,2023-10-05,Failed,103,445.37,2,RIDER9994
9994,ORDER9995,2023-03-01,Failed,93,357.38,5,RIDER9995
9996,ORDER9997,2023-10-13,Failed,68,117.75,2,RIDER9997


**检查这些失败订单的评分是否异常（例如大于3分）**

In [14]:
# 在失败的订单中筛选评分>3的订单
inconsistent_ratings = failed_orders[(failed_orders['Customer_Rating'] > 3)]
print("状态为失败但评分较高（>3）的订单：")
# 输出结果
inconsistent_ratings

状态为失败但评分较高（>3）的订单：


,Order_ID,Date,Delivery_Status,Delivery_Time,Delivery_Cost,Customer_Rating,Rider_ID
0,ORDER1,2023-09-27,Failed,119,64.46,4,RIDER1
1,ORDER2,2023-08-21,Failed,73,94.63,4,RIDER2
26,ORDER27,2023-12-27,Failed,41,193.65,4,RIDER27
34,ORDER35,2023-09-13,Failed,98,429.42,5,RIDER35
52,ORDER53,2023-05-22,Failed,61,256.08,5,RIDER53
...,...,...,...,...,...,...,...
9969,ORDER9970,2023-04-14,Failed,73,351.34,4,RIDER9970
9975,ORDER9976,2023-10-16,Failed,18,387.88,4,RIDER9976
9982,ORDER9983,2023-03-13,Failed,40,56.09,4,RIDER9983
9991,ORDER9992,2023-09-28,Failed,83,63.14,4,RIDER9992


1. 核心发现：数据存在明显逻辑矛盾
 
筛选出了1311条「订单状态为Failed（失败）但用户评分>3分（含4/5分好评）」的记录，这在业务逻辑上是不合理的——正常情况下，订单失败用户很难给出高分评价，说明数据质量存在问题。
 
2. 异常的可能原因
 
- 数据同步/录入错误：订单状态更新与评分字段不同步，或多表关联时匹配出错，导致状态与评分逻辑冲突。

- 系统默认值干扰：评分字段可能保留了默认高分，未随订单失败状态更新。

- 业务特殊场景待确认：极少数情况下可能存在“订单标记失败但用户实际体验良好”的特殊场景，需和业务方核实。
 
3. 下一步可落地的行动
 
1. 先做快速统计：查看失败订单的评分分布，判断是普遍问题还是个别异常。

2. 交叉验证找规律：检查这些异常订单的配送时长、成本是否也存在共性问题，辅助判断是状态错误还是脏数据。

3. 按场景处理数据：

- 若确认是脏数据：直接剔除或标记为无效数据。

- 若存在特殊业务逻辑：单独归类，不纳入常规分析。

**检查是否有失败的订单时长为0或负数**

In [15]:
# 在失败的订单中筛选时长为0或负数的订单
inconsistent_times = failed_orders[failed_orders['Delivery_Time'] <= 0]
print("状态为失败但配送时长<=0的订单：")
# 输出结果
inconsistent_times

状态为失败但配送时长<=0的订单：


,Order_ID,Date,Delivery_Status,Delivery_Time,Delivery_Cost,Customer_Rating,Rider_ID


**从输出结果来看，无失败的订单时长为0或负数。**

疑点：异常的配送时长

逻辑：正常的配送时长通常在几分钟到几小时之间（假设单位是分钟）。如果出现极端的异常值（比如几千分钟，或者负数），就属于不一致。

检查方法：Python

**查看极值**

In [16]:
# 对Delivery_Time列求最大值
print("配送时长最大值:", original_delivery_data['Delivery_Time'].max())
# 对Delivery_Time列求最小值
print("配送时长最小值:", original_delivery_data['Delivery_Time'].min())

配送时长最大值: 120
配送时长最小值: 10


**筛选出明显不合理的时长（例如超过24 * 60=1440分钟，或者小于0）**

In [18]:
# 筛选配送时长超过24h或小于0的订单
weird_times = original_delivery_data[(original_delivery_data['Delivery_Time'] > 1440) | (original_delivery_data['Delivery_Time'] < 0)]
print("时长异常的订单：")
# 输出结果
weird_times

时长异常的订单：


,Order_ID,Date,Delivery_Status,Delivery_Time,Delivery_Cost,Customer_Rating,Rider_ID


**从输出结果来看，无时长异常的订单。**

2.格式不一致（拼写混乱）

虽然 info()显示没有空值，但可能存在文本格式不统一的问题。

疑点：Delivery_Status的拼写变体

逻辑：虽然样本里看起来都是标准的英文，但在大批量数据中，经常会出现 "delivered"（小写）、"Delivered "（带空格）、"Fail"（简写）等变体。

检查方法：Python

**查看该列有多少种不同的值**

In [20]:
print("Delivery_Status 的所有取值：")
original_delivery_data['Delivery_Status'].unique()

Delivery_Status 的所有取值：


<StringArray>
['Failed', 'In Transit', 'Delivered']
Length: 3, dtype: str

**从输出结果来看，Delivery_Status列无大小写错乱、无首尾空格脏数据。**

**保险起见，我们再调用`value_counts`检查一下。**

In [21]:
print("Delivery_Status 的所有取值及个数：")
original_delivery_data['Delivery_Status'].value_counts()

Delivery_Status 的所有取值及个数：


Delivery_Status
Failed        3335
In Transit    3333
Delivered     3332
Name: count, dtype: int64

疑点：Rider_ID的格式

逻辑：样本里有的是 RIDER9577，有的是 RIDER9577后面可能有空格，或者前缀大小写不一。

检查方法：Python

**检查 Rider_ID 的唯一值（注意：如果数据量大，这行可能会打印很多，建议只看前几个）**

In [22]:
unique_riders = original_delivery_data['Rider_ID'].unique()
print(f"骑手ID共有 {len(unique_riders)} 个种类。")

骑手ID共有 10000 个种类。


**从输出结果来看，Rider_ID列的数据都是唯一的。**

**如果有必要，可以检查是否有空格**

In [24]:
original_delivery_data['Rider_ID'].str.strip().unique()

<StringArray>
[    'RIDER1',     'RIDER2',     'RIDER3',     'RIDER4',     'RIDER5',
     'RIDER6',     'RIDER7',     'RIDER8',     'RIDER9',    'RIDER10',
 ...
  'RIDER9991',  'RIDER9992',  'RIDER9993',  'RIDER9994',  'RIDER9995',
  'RIDER9996',  'RIDER9997',  'RIDER9998',  'RIDER9999', 'RIDER10000']
Length: 10000, dtype: str

**Rider_ID列数据全部唯一，无重复；经过strip去空格后依然10000个唯一值 → 该列没有前后空格脏数据。**

3.关联不一致（跨列逻辑）

疑点：极端高成本伴随极端差评

逻辑：Delivery_Cost（配送费）和 Customer_Rating（评分）之间通常应该有某种负相关。如果某笔订单配送费极高（比如几百块），但顾客给了极低的分（1分），这可能暗示服务存在严重问题，或者数据记录有误（比如成本录错了）。

检查方法：Python

**找出费用很高但评分很低的订单**

In [25]:
high_cost_low_rating = original_delivery_data[(original_delivery_data['Delivery_Cost'] > 400) & (original_delivery_data['Customer_Rating'] < 2)]
print("高费用且低评分的异常订单：")
high_cost_low_rating

高费用且低评分的异常订单：


,Order_ID,Date,Delivery_Status,Delivery_Time,Delivery_Cost,Customer_Rating,Rider_ID
3,ORDER4,2023-01-17,In Transit,83,463.06,1,RIDER4
15,ORDER16,2023-07-27,Delivered,118,419.96,1,RIDER16
27,ORDER28,2023-04-05,Failed,87,493.50,1,RIDER28
61,ORDER62,2023-05-11,Failed,35,429.05,1,RIDER62
72,ORDER73,2023-10-20,Delivered,29,433.61,1,RIDER73
...,...,...,...,...,...,...,...
9910,ORDER9911,2023-02-07,Failed,18,456.37,1,RIDER9911
9911,ORDER9912,2023-01-24,Delivered,33,470.05,1,RIDER9912
9918,ORDER9919,2023-01-27,Failed,116,491.31,1,RIDER9919
9923,ORDER9924,2023-11-19,In Transit,44,467.47,1,RIDER9924


**评估不一致数据阶段仅为个人见解，由于不清楚具体数据情况，所以这部分不做清洗处理。**

### 评估无效或错误数据

**可以通过DataFrame的`describe`方法，对数值统计信息进行快速了解。**

In [26]:
original_delivery_data.describe()

,Delivery_Time,Delivery_Cost,Customer_Rating
count,10000.000000,10000.000000,10000.000000
mean,65.277100,273.765728,3.004600
std,31.893934,129.565985,1.414701
min,10.000000,50.010000,1.000000
25%,38.000000,161.737500,2.000000
50%,65.000000,273.870000,3.000000
75%,93.000000,383.672500,4.000000
max,120.000000,500.000000,5.000000


**1. 三个数值字段均无缺失值（count全10000）；**

**2. 配送时长、运费、客户评分的最大/最小值、四分位数全部符合业务常识，不存在异常错误数据（负数、超范围极值）；**

**3. 客户评分集中在2 ~ 4分，整体平均3分；配送时长大多集中在38 ~ 93分钟，运费大多162 ~ 384；**

**4. 从统计层面判断：本数据集无无效、异常错误数据。**

**检查数值列的异常值**

In [29]:
# 检查 Delivery_Time（配送时长）是否为负数或0
negative_time = original_delivery_data[original_delivery_data['Delivery_Time'] <= 0]
print("配送时长 <= 0 的行数:", len(negative_time))

# 检查 Delivery_Cost（配送费）是否为负数
negative_cost = original_delivery_data[original_delivery_data['Delivery_Cost'] < 0]
print("配送费 < 0 的行数:", len(negative_cost))

# 检查 Customer_Rating（客户评分）是否在合理范围（假设 1-5）
invalid_rating = original_delivery_data[(original_delivery_data['Customer_Rating'] < 1) | (original_delivery_data['Customer_Rating'] > 5)]
print("评分不在 1-5 范围内的行数:", len(invalid_rating))

配送时长 <= 0 的行数: 0
配送费 < 0 的行数: 0
评分不在 1-5 范围内的行数: 0


**从输出结果来看，数值列无异常值。**

# 清理数据

**根据前面评估部分得到的结论，我们需要进行的数据清理包括：**
- 将`Date`变量的数据类型转换为datetime
- 将`Delivery_Status`变量的数据类型转换为category（节省内存）
- 将`Customer_Rating`变量的数据类型转换为int8（节省内存）

**为了区分开经过清理的数据和原始的数据，我们创建新的变量cleaned_delivery_data，让它为original_delivery_data复制出的副本。我们之后的清理步骤都将被运用在cleaned_data 上。**

In [3]:
cleaned_delivery_data = original_delivery_data.copy()
cleaned_delivery_data.head()

,Order_ID,Date,Delivery_Status,Delivery_Time,Delivery_Cost,Customer_Rating,Rider_ID
0,ORDER1,2023-09-27,Failed,119,64.46,4,RIDER1
1,ORDER2,2023-08-21,Failed,73,94.63,4,RIDER2
2,ORDER3,2023-11-22,Failed,49,324.06,3,RIDER3
3,ORDER4,2023-01-17,In Transit,83,463.06,1,RIDER4
4,ORDER5,2023-11-28,Delivered,21,469.34,5,RIDER5


**2.1转换数据类型**

In [4]:
# 将 Date 从 str 转为 datetime
cleaned_delivery_data['Date'] = pd.to_datetime(cleaned_delivery_data['Date'])

# 将 Delivery_Status 转为 category（节省内存）
cleaned_delivery_data['Delivery_Status'] = cleaned_delivery_data['Delivery_Status'].astype('category')

# 将 Customer_Rating 从 int64 转为 int8（节省内存）
cleaned_delivery_data['Customer_Rating'] = cleaned_delivery_data['Customer_Rating'].astype('int8')

# 可选：将 Rider_ID 也转为 category（如果骑手数量远小于订单数）
# cleaned_data['Rider_ID'] = cleaned_data['Rider_ID'].astype('category')

print("清洗后数据类型:\n", cleaned_delivery_data.dtypes)

清洗后数据类型:
 Order_ID                      str
Date               datetime64[us]
Delivery_Status          category
Delivery_Time               int64
Delivery_Cost             float64
Customer_Rating              int8
Rider_ID                      str
dtype: object


# 保存清洗后的数据

In [5]:
cleaned_delivery_data.to_csv("D:/delivery-data-analysis-main/cleaned_delivery_data.csv")